In [72]:
from ortools.sat.python import cp_model

In [73]:
model = cp_model.CpModel()

In [74]:
num_vals = 8
brown = model.NewIntVar(0, num_vals - 1, "brown")
red = model.NewIntVar(0, num_vals - 1, "red")
green = model.NewIntVar(0, num_vals - 1, "green")
yellow = model.NewIntVar(0, num_vals - 1, "yellow")
green2 = model.NewIntVar(0, num_vals - 1, "green2")
blue = model.NewIntVar(0, num_vals - 1, "blue")

In [75]:
model.AddAllDifferent([brown, red, green, yellow, green2, blue])

# earth rot chamber
# 8 presses is the max (either required or will result in punishment)
# blue immediately after green or red (but not both?)
# yellow precedes green (green is maybe last?)

# blood red chamber
# brown precedes red
# green by green (on a red bench - more colorblind hints?)

# soulbane chamber
# red before blue but suggests it should be reversed blue before red?
# green before yellow

# blue vein chamber
# green after red
# first occurs once, last twice

# colors referenced ?
# blue, (green|red), yellow, green, brown, red, red, blue, green, yellow, green, red

# blue 3
# brown 1
# green 4
# red 5
# yellow 2

# model.Add(green < blue)
# model.Add(red < blue)
model.Add(yellow + 1 == green)
model.Add(brown + 1 == red)
model.Add(green + 1 == green2)
model.Add(red + 1 == blue)
model.Add(red < green)



In [76]:
solver = cp_model.CpSolver()
status = solver.Solve(model)

In [77]:
if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"status: {status}")
    print(f"brown = {solver.Value(brown)}")
    print(f"red = {solver.Value(red)}")
    print(f"green = {solver.Value(green)}")
    print(f"yellow = {solver.Value(yellow)}")
    print(f"green2 = {solver.Value(green2)}")
    print(f"blue = {solver.Value(blue)}")
else:
    print("No solution found.")

status: 4
brown = 0
red = 1
green = 6
yellow = 5
green2 = 7
blue = 2


In [3]:
from z3 import *

# Define color indices and names
colors = {
    'Red': 0,
    'Yellow': 1,
    'Green': 2,
    'Blue': 3,
    'Brown': 4
}
color_names = ['Red', 'Yellow', 'Green', 'Blue', 'Brown']
n = 8  # number of button presses

# Generate a new solver instance
def create_solver():
    seq = [Int(f'seq_{i}') for i in range(n)]
    solver = Solver()

    # Add requirement that each collor occurs at least once
    for cid in range(5):
        count = Sum([If(s == cid, 1, 0) for s in seq])
        solver.add(count >= 1)
    
    # Valid color range
    for s in seq:
        solver.add(s >= 0, s <= 4)

    # First color occurs only once
    first = seq[0]
    for cid in range(5):
        count = Sum([If(s == cid, 1, 0) for s in seq])
        solver.add(Implies(first == cid, count == 1))

    # Last color occurs exactly twice
    last = seq[-1]
    for cid in range(5):
        count = Sum([If(s == cid, 1, 0) for s in seq])
        solver.add(Implies(last == cid, count == 2))

    # Brown before Red
    for i in range(n):
        for j in range(n):
            solver.add(Implies(And(seq[i] == colors['Red'], seq[j] == colors['Brown']), j < i))

    # Red before Green
    for i in range(n):
        for j in range(n):
            solver.add(Implies(And(seq[i] == colors['Green'], seq[j] == colors['Red']), j < i))

    # Green before Yellow
    for i in range(n):
        for j in range(n):
            solver.add(Implies(And(seq[i] == colors['Yellow'], seq[j] == colors['Green']), j < i))

    # Green followed immediately by Blue at least once
    green_blue_pairs = [And(seq[i] == colors['Green'], seq[i+1] == colors['Blue']) for i in range(n - 1)]
    solver.add(Or(green_blue_pairs))

    # "Green by Green": adjacent greens
    green_adjacent = [And(seq[i] == colors['Green'], seq[i+1] == colors['Green']) for i in range(n - 1)]
    solver.add(Or(green_adjacent))

    return solver, seq

# Collect all solutions
def get_all_solutions():
    all_sequences = []
    solver, seq = create_solver()
    while solver.check() == sat:
        model = solver.model()
        current = [model.evaluate(seq[i]).as_long() for i in range(n)]
        all_sequences.append(current)

        # Block this solution to find the next one
        solver.add(Or([seq[i] != current[i] for i in range(n)]))

    # Convert indices to color names and sort for consistency
    def to_color_names(indices):
        return [color_names[i] for i in indices]

    all_sequences_named = [to_color_names(seq) for seq in all_sequences]
    all_sequences_named.sort()
    return all_sequences_named

# Run and print all sequences
solutions = get_all_solutions()
for idx, sol in enumerate(solutions):
    print(f"{idx+1}: {sol}")
print(f"Total valid sequences found: {len(solutions)}")


1: ['Brown', 'Blue', 'Red', 'Green', 'Green', 'Blue', 'Yellow', 'Yellow']
2: ['Brown', 'Red', 'Blue', 'Green', 'Green', 'Blue', 'Yellow', 'Yellow']
3: ['Brown', 'Red', 'Green', 'Blue', 'Green', 'Green', 'Yellow', 'Blue']
4: ['Brown', 'Red', 'Green', 'Blue', 'Green', 'Green', 'Yellow', 'Yellow']
5: ['Brown', 'Red', 'Green', 'Green', 'Blue', 'Blue', 'Yellow', 'Yellow']
6: ['Brown', 'Red', 'Green', 'Green', 'Blue', 'Green', 'Yellow', 'Blue']
7: ['Brown', 'Red', 'Green', 'Green', 'Blue', 'Green', 'Yellow', 'Yellow']
8: ['Brown', 'Red', 'Green', 'Green', 'Blue', 'Yellow', 'Blue', 'Yellow']
9: ['Brown', 'Red', 'Green', 'Green', 'Blue', 'Yellow', 'Yellow', 'Blue']
10: ['Brown', 'Red', 'Green', 'Green', 'Green', 'Blue', 'Yellow', 'Blue']
11: ['Brown', 'Red', 'Green', 'Green', 'Green', 'Blue', 'Yellow', 'Yellow']
12: ['Brown', 'Red', 'Red', 'Green', 'Green', 'Blue', 'Yellow', 'Blue']
13: ['Brown', 'Red', 'Red', 'Green', 'Green', 'Blue', 'Yellow', 'Yellow']
Total valid sequences found: 13
